# 🎬 CineMind AI — Hybrid Movie Recommender

A hybrid (content-based + collaborative filtering) movie recommender built on the MovieLens dataset, with a fully custom, animated Gradio UI.

**How to run:**
1. Run **Cell 1**. It installs packages and then automatically restarts the Colab runtime (you'll see it disconnect/reconnect — that's expected, not an error).
2. After it reconnects, run every remaining cell top to bottom. **Don't re-run Cell 1.**
3. The last cell launches the app with a public share link.

In [ ]:
# Installs the latest compatible Gradio, then force-restarts the runtime.
# This is required because Colab pre-loads its own versions of fastapi /
# pydantic / typing_extensions, which silently conflict with Gradio unless
# we start from a clean process. After this runs, the runtime reconnects
# automatically — continue from the NEXT cell, don't re-run this one.

!pip install -q --upgrade gradio pandas numpy scikit-learn scipy matplotlib

import os
os.kill(os.getpid(), 9)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 34.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.


In [2]:
import os
import zipfile
import urllib.request
import ssl

DATA_DIR = "ml-latest-small"
ZIP_PATH = "ml-latest-small.zip"
URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

if not os.path.exists(DATA_DIR):
    print("Downloading MovieLens dataset...")

    # Temporary workaround for certificate verification error
    context = ssl._create_unverified_context()

    with urllib.request.urlopen(URL, context=context) as response:
        with open(ZIP_PATH, "wb") as f:
            f.write(response.read())

    print("Download complete.")

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")

    print("Done.")

else:
    print("Dataset already present.")

Download complete.
Done.


In [3]:
import re
import numpy as np
import pandas as pd

movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")
tags = pd.read_csv(f"{DATA_DIR}/tags.csv")

# Keep only movies that have at least one rating (needed for collaborative filtering)
valid_ids = set(ratings["movieId"].unique())
movies = movies[movies["movieId"].isin(valid_ids)].reset_index(drop=True)

def extract_year(title):
    m = re.search(r"\((\d{4})\)\s*$", title)
    return int(m.group(1)) if m else None

movies["year"] = movies["title"].apply(extract_year)
movies["clean_title"] = movies["title"].apply(lambda t: re.sub(r"\s*\(\d{4}\)\s*$", "", t).strip())
movies["genre_list"] = movies["genres"].apply(lambda g: [] if g == "(no genres listed)" else g.split("|"))

# Aggregate free-text tags per movie (adds flavor to the content-based model)
tags_agg = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x.astype(str))).reset_index()
tags_agg.columns = ["movieId", "tag_text"]
movies = movies.merge(tags_agg, on="movieId", how="left")
movies["tag_text"] = movies["tag_text"].fillna("")

# Rating stats per movie
rating_stats = ratings.groupby("movieId")["rating"].agg(["mean", "count"]).reset_index()
rating_stats.columns = ["movieId", "avg_rating", "num_ratings"]
movies = movies.merge(rating_stats, on="movieId", how="left")
movies["avg_rating"] = movies["avg_rating"].fillna(0).round(2)
movies["num_ratings"] = movies["num_ratings"].fillna(0).astype(int)

# IMPORTANT: movies is now final — its row order defines the index used by
# every similarity matrix built below. Do not reorder/resample this dataframe
# afterwards, or the similarity matrices will point at the wrong rows.
movies = movies.reset_index(drop=True)
movies["idx"] = movies.index

movieid_to_idx = dict(zip(movies["movieId"], movies["idx"]))
title_to_idx = dict(zip(movies["clean_title"].str.lower(), movies["idx"]))

print(f"Loaded {len(movies)} movies and {len(ratings)} ratings.")
movies.head()


Loaded 9724 movies and 100836 ratings.


,movieId,title,genres,year,clean_title,genre_list,tag_text,avg_rating,num_ratings,idx
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",pixar pixar fun,3.92,215,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"[Adventure, Children, Fantasy]",fantasy magic board game Robin Williams game,3.43,110,1
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"[Comedy, Romance]",moldy old,3.26,52,2
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,"[Comedy, Drama, Romance]",,2.36,7,3
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,[Comedy],pregnancy remake,3.07,49,4


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies["content_soup"] = (
    movies["genre_list"].apply(lambda g: " ".join(g).replace("-", "")) + " " + movies["tag_text"]
)

tfidf = TfidfVectorizer(stop_words="english", min_df=1)
tfidf_matrix = tfidf.fit_transform(movies["content_soup"])

content_sim = cosine_similarity(tfidf_matrix).astype(np.float32)
print("Content similarity matrix:", content_sim.shape)


Content similarity matrix: (9724, 9724)


In [5]:
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

ratings_f = ratings[ratings["movieId"].isin(movieid_to_idx.keys())].copy()

user_ids = ratings_f["userId"].unique()
userid_to_uidx = {u: i for i, u in enumerate(user_ids)}

row_idx = ratings_f["userId"].map(userid_to_uidx).values
col_idx = ratings_f["movieId"].map(movieid_to_idx).values
data_vals = ratings_f["rating"].values

n_users = len(user_ids)
n_movies = len(movies)

user_item = csr_matrix((data_vals, (row_idx, col_idx)), shape=(n_users, n_movies))

n_components = min(50, min(user_item.shape) - 1)
svd = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd.fit_transform(user_item)
item_factors = svd.components_.T  # (n_movies, n_components)

collab_sim = cosine_similarity(item_factors).astype(np.float32)
print("Collaborative similarity matrix:", collab_sim.shape)
print(f"Explained variance (SVD, {n_components} comps): {svd.explained_variance_ratio_.sum():.2%}")


Collaborative similarity matrix: (9724, 9724)
Explained variance (SVD, 50 comps): 53.88%


In [6]:
import random

def find_movie_idx(title):
    key = title.strip().lower()
    if key in title_to_idx:
        return title_to_idx[key]
    matches = movies[movies["clean_title"].str.lower().str.contains(re.escape(key), regex=True)]
    if len(matches):
        return int(matches.iloc[0]["idx"])
    return None

def hybrid_recommend(title, top_n=10, content_weight=0.5, min_votes=1):
    """Blend content-based and collaborative similarity for a given movie title."""
    idx = find_movie_idx(title)
    if idx is None:
        return None, f"Couldn't find a movie matching '{title}'. Try another title."

    sim = content_weight * content_sim[idx] + (1 - content_weight) * collab_sim[idx]
    order = np.argsort(-sim)

    results = []
    for i in order:
        i = int(i)
        if i == idx:
            continue
        if movies.iloc[i]["num_ratings"] < min_votes:
            continue
        results.append(i)
        if len(results) >= top_n:
            break

    base_title = movies.iloc[idx]["clean_title"]
    rows = [(movies.iloc[i], f"Similar to {base_title}") for i in results]
    return rows, None

MOOD_GENRES = {
    "😄 Happy": ["Comedy", "Animation", "Adventure"],
    "😢 Sad / Reflective": ["Drama"],
    "🔥 Excited": ["Action", "Thriller", "Sci-Fi"],
    "💕 Romantic": ["Romance"],
    "😱 Scared": ["Horror", "Mystery"],
    "🤯 Mind-Bending": ["Sci-Fi", "Mystery", "Film-Noir"],
    "🌙 Chill / Relaxed": ["Documentary", "Animation", "Family"],
    "😂 Ridiculous": ["Comedy"],
}

def mood_recommend(mood, top_n=10, min_votes=10):
    genres = MOOD_GENRES.get(mood, [])
    mask = movies["genre_list"].apply(lambda gl: any(g in gl for g in genres))
    pool = movies[mask & (movies["num_ratings"] >= min_votes)].copy()
    if pool.empty:
        pool = movies[mask].copy()
    if pool.empty:
        return []

    pool["score"] = pool["avg_rating"] * np.log1p(pool["num_ratings"])
    pool = pool.sort_values("score", ascending=False).head(max(top_n * 3, top_n))
    n_sample = min(top_n, len(pool))
    pool = pool.sample(n=n_sample, random_state=random.randint(0, 10_000))

    rows = [(r, f"Matches your {mood} mood") for _, r in pool.iterrows()]
    return rows

def surprise_me(min_votes=20):
    pool = movies[movies["num_ratings"] >= min_votes].copy()
    if pool.empty:
        pool = movies.copy()

    weights = pool["avg_rating"] * np.log1p(pool["num_ratings"])
    weights = weights.replace(0, 0.01)  # avoid an all-zero weight vector
    pick = pool.sample(n=1, weights=weights).iloc[0]

    similar_rows, _ = hybrid_recommend(pick["clean_title"], top_n=6)
    return pick, similar_rows

print("Recommendation engine ready.")


Recommendation engine ready.


In [7]:
import urllib.parse

GRADIENTS = [
    "linear-gradient(135deg,#f6d365,#fda085)",
    "linear-gradient(135deg,#a18cd1,#fbc2eb)",
    "linear-gradient(135deg,#84fab0,#8fd3f4)",
    "linear-gradient(135deg,#ff9a9e,#fecfef)",
    "linear-gradient(135deg,#30cfd0,#330867)",
    "linear-gradient(135deg,#fbc2eb,#a6c1ee)",
    "linear-gradient(135deg,#ff6a00,#ee0979)",
    "linear-gradient(135deg,#00c6ff,#0072ff)",
    "linear-gradient(135deg,#f7971e,#ffd200)",
    "linear-gradient(135deg,#f857a6,#ff5858)",
    "linear-gradient(135deg,#43cea2,#185a9d)",
    "linear-gradient(135deg,#7f00ff,#e100ff)",
]

def gradient_for(seed_text):
    h = sum(ord(c) for c in str(seed_text))
    return GRADIENTS[h % len(GRADIENTS)]

def star_html(rating):
    rating = max(0.0, min(5.0, float(rating)))
    full = int(rating)
    half = 1 if (rating - full) >= 0.5 else 0
    empty = 5 - full - half
    return "★" * full + ("½" if half else "") + "☆" * empty

def build_info_links(clean_title, year):
    """One-click links: a Google search for full details, and JustWatch to
    find *legal* streaming/rental sources. No scraping, no unlicensed
    streaming sites — just search links that always work."""
    query_full = urllib.parse.quote_plus(f"{clean_title} {year or ''} movie".strip())
    google_url = f"https://www.google.com/search?q={query_full}"
    query_title = urllib.parse.quote_plus(clean_title)
    justwatch_url = f"https://www.justwatch.com/us/search?q={query_title}"
    imdb_url = f"https://www.imdb.com/find/?q={query_title}"
    return google_url, justwatch_url, imdb_url

def render_card(row, reason=None):
    grad = gradient_for(row["clean_title"])
    genre_list = row["genre_list"][:4] if row["genre_list"] else []
    genres = "".join(f"<span class='chip'>{g}</span>" for g in genre_list) or "<span class='chip'>Uncategorized</span>"
    year = f"({row['year']})" if row["year"] else ""
    reason_html = f"<div class='reason'>✨ {reason}</div>" if reason else ""
    title = str(row["clean_title"]).replace("<", "&lt;").replace(">", "&gt;")

    google_url, justwatch_url, imdb_url = build_info_links(row["clean_title"], row["year"])
    links_html = f"""
      <div class='card-links'>
        <a href='{google_url}' target='_blank' rel='noopener noreferrer' class='link-btn'>🔍 Full Info</a>
        <a href='{imdb_url}' target='_blank' rel='noopener noreferrer' class='link-btn'>🎞 IMDb</a>
        <a href='{justwatch_url}' target='_blank' rel='noopener noreferrer' class='link-btn'>▶ Where to Watch</a>
      </div>
    """

    return f"""
    <div class='movie-card' style='background:{grad}'>
      <div class='card-top'>
        <div class='card-title'>{title} <span class='card-year'>{year}</span></div>
        <div class='card-rating'>{star_html(row['avg_rating'])}<span class='rating-num'>{row['avg_rating']}</span></div>
      </div>
      <div class='card-genres'>{genres}</div>
      <div class='card-votes'>{int(row['num_ratings'])} ratings</div>
      {reason_html}
      {links_html}
    </div>
    """

def render_grid(rows_with_reasons, empty_message="No movies matched — try loosening the filters."):
    if not rows_with_reasons:
        return f"<div class='empty-state'>{empty_message}</div>"
    cards = "".join(render_card(r, reason) for r, reason in rows_with_reasons)
    return f"<div class='card-grid'>{cards}</div>"

print("UI helpers ready.")


UI helpers ready.


In [ ]:
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ALL_TITLES = movies.sort_values("num_ratings", ascending=False)["clean_title"].tolist()

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;800&family=Space+Grotesk:wght@500;700&display=swap');

.gradio-container {
    font-family: 'Poppins', sans-serif !important;
    background: radial-gradient(circle at top left, #1b1035, #05010a 70%) !important;
}

#hero-title {
    font-family: 'Space Grotesk', sans-serif;
    font-size: 2.6rem;
    font-weight: 800;
    text-align: center;
    background: linear-gradient(90deg, #ff6a00, #ee0979, #00c6ff, #7f00ff);
    background-size: 300% 300%;
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    animation: gradientShift 6s ease infinite;
    margin-bottom: 0;
    padding-top: 10px;
}

@keyframes gradientShift {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

#hero-sub {
    text-align: center;
    color: #c7c7d9;
    margin-top: 4px;
    margin-bottom: 18px;
    font-size: 0.95rem;
}

.card-grid {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(230px, 1fr));
    gap: 18px;
    margin-top: 10px;
}

.movie-card {
    border-radius: 18px;
    padding: 16px;
    color: #14141d;
    box-shadow: 0 8px 24px rgba(0,0,0,0.35);
    transition: transform 0.25s ease, box-shadow 0.25s ease;
    position: relative;
    overflow: hidden;
}

.movie-card:hover {
    transform: translateY(-6px) scale(1.02) rotate(-0.4deg);
    box-shadow: 0 16px 34px rgba(0,0,0,0.5);
}

.card-top { display: flex; justify-content: space-between; align-items: flex-start; gap: 8px; }
.card-title { font-weight: 800; font-size: 1.02rem; line-height: 1.25; }
.card-year { font-weight: 500; opacity: 0.7; font-size: 0.82rem; }
.card-rating { font-weight: 700; white-space: nowrap; font-size: 0.95rem; }
.rating-num { font-size: 0.72rem; opacity: 0.8; margin-left: 4px; }
.card-genres { margin-top: 10px; display: flex; flex-wrap: wrap; gap: 6px; }

.chip {
    background: rgba(255,255,255,0.4);
    padding: 2px 10px;
    border-radius: 999px;
    font-size: 0.7rem;
    font-weight: 700;
}

.card-votes { margin-top: 8px; font-size: 0.72rem; opacity: 0.75; }

.reason {
    margin-top: 10px;
    font-size: 0.75rem;
    font-weight: 700;
    background: rgba(255,255,255,0.45);
    padding: 4px 10px;
    border-radius: 10px;
    display: inline-block;
}

.card-links {
    margin-top: 12px;
    display: flex;
    gap: 6px;
    flex-wrap: wrap;
}

.link-btn {
    text-decoration: none !important;
    background: rgba(20,20,29,0.78);
    color: #ffffff !important;
    padding: 5px 10px;
    border-radius: 10px;
    font-size: 0.68rem;
    font-weight: 700;
    transition: background 0.2s ease, transform 0.2s ease;
}

.link-btn:hover {
    background: rgba(20,20,29,1);
    transform: translateY(-2px);
}

.empty-state {
    text-align: center;
    color: #c7c7d9;
    padding: 50px 20px;
    font-size: 1rem;
}
"""

def ui_discover(title, top_n, content_weight):
    if not title:
        return "<div class='empty-state'>Pick or type a movie title to get started ✨</div>"
    rows, err = hybrid_recommend(title, top_n=int(top_n), content_weight=float(content_weight), min_votes=1)
    if err:
        return f"<div class='empty-state'>{err}</div>"
    return render_grid(rows)

def ui_mood(mood, top_n):
    rows = mood_recommend(mood, top_n=int(top_n))
    return render_grid(rows)

def ui_surprise():
    pick, similar_rows = surprise_me()
    pick_reason = "Today's pick for you"
    hero = f"<div class='card-grid'>{render_card(pick, pick_reason)}</div>"
    rest = render_grid(similar_rows) if similar_rows else ""
    footer = "<h3 style='color:#fff;margin-top:24px;'>Because you might like this too...</h3>" if similar_rows else ""
    return hero + footer + rest

def ui_analytics():
    top_genres = pd.Series([g for gl in movies["genre_list"] for g in gl]).value_counts().head(12)

    fig1, ax1 = plt.subplots(figsize=(6, 4))
    fig1.patch.set_facecolor("#0f0f1a")
    ax1.set_facecolor("#0f0f1a")
    ax1.barh(top_genres.index[::-1], top_genres.values[::-1], color="#ee0979")
    ax1.set_title("Top Genres by Movie Count", color="white")
    ax1.tick_params(colors="white")
    for spine in ax1.spines.values():
        spine.set_color("white")
    fig1.tight_layout()

    top_movies = movies[movies["num_ratings"] >= 50].sort_values("avg_rating", ascending=False).head(10)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    fig2.patch.set_facecolor("#0f0f1a")
    ax2.set_facecolor("#0f0f1a")
    ax2.barh(top_movies["clean_title"][::-1], top_movies["avg_rating"][::-1], color="#00c6ff")
    ax2.set_title("Top Rated Movies (50+ ratings)", color="white")
    ax2.set_xlim(0, 5)
    ax2.tick_params(colors="white")
    for spine in ax2.spines.values():
        spine.set_color("white")
    fig2.tight_layout()

    return fig1, fig2

with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Base(primary_hue="purple", secondary_hue="pink"), title="CineMind AI") as demo:
    gr.HTML(
        "<div id='hero-title'>🎬 CineMind AI</div>"
        "<div id='hero-sub'>A hybrid AI movie recommender — content + collaborative filtering, "
        "mood matching, and a chaos-good UI</div>"
    )

    with gr.Tab("🔎 Discover"):
        with gr.Row():
            title_input = gr.Dropdown(
                choices=ALL_TITLES,
                value=ALL_TITLES[0] if ALL_TITLES else None,
                label="Search a movie you love",
                allow_custom_value=True,
                filterable=True,
                scale=3,
            )
            top_n_1 = gr.Slider(4, 20, value=10, step=1, label="How many recs?", scale=1)
        content_weight = gr.Slider(
            0, 1, value=0.5, step=0.05,
            label="⬅ Collaborative (taste-based)   —   Content-based (genre/tag-based) ➡",
        )
        discover_btn = gr.Button("✨ Recommend", variant="primary")
        discover_out = gr.HTML()
        discover_btn.click(ui_discover, inputs=[title_input, top_n_1, content_weight], outputs=discover_out)

    with gr.Tab("🎭 Mood Match"):
        mood_input = gr.Radio(list(MOOD_GENRES.keys()), label="What's your mood tonight?", value="😄 Happy")
        top_n_2 = gr.Slider(4, 20, value=10, step=1, label="How many recs?")
        mood_btn = gr.Button("🎯 Match My Mood", variant="primary")
        mood_out = gr.HTML()
        mood_btn.click(ui_mood, inputs=[mood_input, top_n_2], outputs=mood_out)

    with gr.Tab("🎲 Surprise Me"):
        gr.Markdown("Feeling indecisive? Let the algorithm (and a little chaos) choose for you.")
        surprise_btn = gr.Button("🎰 Surprise Me!", variant="primary")
        surprise_out = gr.HTML()
        surprise_btn.click(ui_surprise, inputs=None, outputs=surprise_out)

    with gr.Tab("📊 Analytics"):
        gr.Markdown("A peek behind the curtain at the dataset powering CineMind AI.")
        analytics_btn = gr.Button("Generate Analytics")
        with gr.Row():
            plot1 = gr.Plot()
            plot2 = gr.Plot()
        analytics_btn.click(ui_analytics, inputs=None, outputs=[plot1, plot2])

demo.queue().launch(share=True, debug=True)


/tmp/ipykernel_862/366736054.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Base(primary_hue="purple", secondary_hue="pink"), title="CineMind AI") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f16e3229dbdc07a57b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
